# 🏛️ Stoic Content Pipeline — Interactive Workbench & Telemetry Dashboard

Welcome to the interactive workbench for the **Stoic Shorts Video Pipeline**.

This notebook provides interactive module execution, quote database analytics, B-Roll vision re-ranking inspection, subtitle word-timing analysis, and system performance telemetry.

---

## ⚙️ 0. System Environment & Dependency Check

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print(f"✅ Python Version: {sys.version.split()[0]}")
print(f"✅ Working Directory: {os.getcwd()}")

# Check for required project files
required_files = ["stoic_quotes_database.csv", "voice_bible.md", "broll_keywords.json", "orchestrator.py"]
for f in required_files:
    status = "FOUND" if os.path.exists(f) else "MISSING"
    print(f"  - [{status}] {f}")

--- 
## 📊 1. Quote Database Analytics & Theme Distribution

In [ ]:
csv_path = "stoic_quotes_database.csv"
if os.path.exists(csv_path):
    df_quotes = pd.read_csv(csv_path)
    print(f"Total Quotes Loaded: {len(df_quotes)}")
    display(df_quotes.head())
else:
    print("❌ stoic_quotes_database.csv not found.")

In [ ]:
# Visualizing Theme & Author Breakdown
if 'df_quotes' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Quotes per Theme
    theme_counts = df_quotes['theme'].value_counts()
    sns.barplot(x=theme_counts.values, y=theme_counts.index, ax=axes[0], palette="viridis")
    axes[0].set_title("Quote Distribution by Stoic Theme")
    axes[0].set_xlabel("Count")
    
    # Subplot 2: Quotes per Author
    author_counts = df_quotes['author'].value_counts()
    sns.barplot(x=author_counts.values, y=author_counts.index, ax=axes[1], palette="magma")
    axes[1].set_title("Quote Distribution by Author")
    axes[1].set_xlabel("Count")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Exhaustion Rate & Length Bucket Analysis
used_quotes_path = "used_quotes.json"
used_count = 0
if os.path.exists(used_quotes_path):
    with open(used_quotes_path, "r", encoding="utf-8") as f:
        used_quotes = json.load(f)
        used_count = len(used_quotes)

total_quotes = len(df_quotes) if 'df_quotes' in locals() else 0
exhaustion_pct = (used_count / total_quotes * 100) if total_quotes > 0 else 0

print(f"📈 Database Usage Status:")
print(f"  - Total Quotes: {total_quotes}")
print(f"  - Used Quotes:  {used_count}")
print(f"  - Remaining:    {total_quotes - used_count}")
print(f"  - Exhaustion Rate: {exhaustion_pct:.1f}%")

--- 
## 🧪 2. Interactive Pipeline Module Runner

In [ ]:
import subprocess

def run_pipeline_step(command):
    print(f"🚀 Running: {command}")
    res = subprocess.run(command, shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"✅ Success! Output:\n{res.stdout[:300]}...")
    else:
        print(f"❌ Failed with exit code {res.returncode}:\n{res.stderr}")

# Interactive Step 1: Quote Picker
run_pipeline_step("python quote_picker.py --theme fear")

In [ ]:
# Display Picked Quote
if os.path.exists("quote.json"):
    with open("quote.json", "r", encoding="utf-8") as f:
        quote_data = json.load(f)
        print("📜 Selected Quote Handoff:")
        print(json.dumps(quote_data, indent=2))
else:
    print("quote.json does not exist yet.")

In [ ]:
# Interactive Step 2: Script Generator (via Gemini 3.6 Flash)
# Note: Requires GEMINI_API_KEY in environment
run_pipeline_step("python script_generator.py")

In [ ]:
# Display Generated Script Beats
if os.path.exists("script.json"):
    with open("script.json", "r", encoding="utf-8") as f:
        script_data = json.load(f)
        print(f"🎙️ Father-Son Script (Word Count: {script_data.get('word_count')}):\n")
        print(f"[HOOK]:        {script_data.get('hook')}")
        print(f"[CONTEXT]:     {script_data.get('context')}")
        print(f"[APPLICATION]: {script_data.get('application')}")
        print(f"[CTA]:         {script_data.get('cta')}")
else:
    print("script.json does not exist yet.")

--- 
## 🎥 3. B-Roll Matcher & Multimodal Vision Re-Ranking Inspector

In [ ]:
# Inspecting B-Roll Manifest
manifest_path = "broll_manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        broll_data = json.load(f)
    
    clips = broll_data.get("clips", [])
    df_clips = pd.DataFrame(clips)
    print(f"Total B-Roll Clips Matched: {len(df_clips)}")
    display(df_clips[["local_path", "resolution", "fallback_tier", "duration", "search_phrase_used"]].head(10))
else:
    print("❌ broll_manifest.json not found.")

In [ ]:
# Resolution & Fallback Tier Breakdown
if 'df_clips' in locals() and not df_clips.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df_clips["resolution"].value_counts().plot(kind="pie", ax=axes[0], autopct="%1.1f%%", colors=sns.color_palette("pastel"))
    axes[0].set_title("Clip Resolution Distribution")
    axes[0].set_ylabel("")
    
    df_clips["fallback_tier"].value_counts().plot(kind="bar", ax=axes[1], color="teal")
    axes[1].set_title("Fallback Tier Distribution")
    axes[1].set_ylabel("Count")
    
    plt.tight_layout()
    plt.show()

--- 
## 🗣️ 4. Subtitle Word-Timing & Audio Analysis

In [ ]:
# Analyzing Audio Duration & Whisper Word Sync
words_json_path = "captions_words.json"
if os.path.exists(words_json_path):
    with open(words_json_path, "r", encoding="utf-8") as f:
        words_data = json.load(f)
    
    df_words = pd.DataFrame(words_data)
    df_words["duration"] = df_words["end"] - df_words["start"]
    
    print(f"Total Transcribed Words: {len(df_words)}")
    print(f"Average Word Speaking Duration: {df_words['duration'].mean():.2f} seconds")
    
    plt.figure(figsize=(10, 4))
    sns.histplot(df_words["duration"], bins=20, kde=True, color="coral")
    plt.title("Speech Cadence: Distribution of Word Speaking Durations")
    plt.xlabel("Duration (seconds)")
    plt.ylabel("Frequency")
    plt.show()
else:
    print("captions_words.json not found. Run module 4 to generate.")

--- 
## 📈 5. Telemetry & Token Consumption Dashboard

In [ ]:
# Parsing Telemetry Run Data
telemetry_path = "telemetry_run.json"
if os.path.exists(telemetry_path):
    with open(telemetry_path, "r", encoding="utf-8") as f:
        telemetry_data = json.load(f)
    
    api_calls = telemetry_data.get("api_calls", [])
    df_telemetry = pd.DataFrame(api_calls)
    
    print("📊 Gemini API Token Telemetry:")
    display(df_telemetry)
    
    if not df_telemetry.empty:
        plt.figure(figsize=(9, 4))
        df_telemetry.plot(x="module", y=["prompt_tokens", "candidates_tokens", "total_tokens"], kind="bar")
        plt.title("Gemini API Token Consumption per Pipeline Module")
        plt.ylabel("Token Count")
        plt.xticks(rotation=0)
        plt.legend(["Prompt Tokens", "Candidate Tokens", "Total Tokens"])
        plt.tight_layout()
        plt.show()
else:
    print("telemetry_run.json not found.")

--- 
### 🏁 Summary & Next Steps
- Use this notebook for **interactive debugging**, **theme analytics**, and **quality auditing**.
- To trigger a complete video generation run, execute `python orchestrator.py` in your terminal or via cell `%run orchestrator.py`.